# Synthetic data from knowledge

References:

* [Nemotron-CC](https://arxiv.org/abs/2412.02595v2)
* [BeyondWeb](https://blog.datologyai.com/beyondweb/)
* [Kimi K2](https://arxiv.org/abs/2507.20534)

In [1]:
from pathlib import Path

OUTPUT_DIR = Path("output/synthetic/v2")
INPUT_ROOT_DIR = Path("output/wiki")
MIN_CHUNK_SIZE = 256
CHUNK_SIZE = 1024

INPUT_DIRS = [
    INPUT_ROOT_DIR,
    INPUT_ROOT_DIR / "fandom",
    INPUT_ROOT_DIR / "fandom/crawl/luoghi",
    INPUT_ROOT_DIR / "fandom/crawl/personaggi",
    INPUT_ROOT_DIR / "fandom/crawl/personaggi/amici",
    INPUT_ROOT_DIR / "fandom/crawl/personaggi/evroniani",
    INPUT_ROOT_DIR / "fandom/crawl/personaggi/nemici",
    INPUT_ROOT_DIR / "fandom/crawl/personaggi/starcorp",
    INPUT_ROOT_DIR / "fandom/crawl/personaggi/xerbiani",
    INPUT_ROOT_DIR / "fandom/crawl/storie/storie-di-pk-pkna",
    INPUT_ROOT_DIR / "fandom/crawl/tecnologia",
    INPUT_ROOT_DIR / "manual",
]
INPUT_DOCS = sorted([path for d in INPUT_DIRS for path in d.glob("*.md")])

INPUT_MODEL_CONVERSATIONS = Path("output/dataset/dspy-dataset.csv")

## Chunking

In [2]:
from markdown_it import MarkdownIt


def split_markdown(md_text: str) -> list[str]:
    md = MarkdownIt()
    tokens = md.parse(md_text)

    chunks = []
    current = []
    for i, tok in enumerate(tokens):
        if tok.type == "heading_open":
            # flush previous chunk
            if current:
                chunks.append("".join(current))
                current = []
        # reconstruct token text
        if tok.type == "inline":
            current.append(tok.content + "\n")
        elif tok.type.endswith("_open") or tok.type.endswith("_close"):
            # keep markup like ###, code fences, list markers
            current.append(tok.markup + "\n")
        elif tok.type == "fence":
            current.append(f"{tok.markup}{tok.info}\n{tok.content}{tok.markup}\n")
        elif tok.content:
            current.append(tok.content + "\n")

    if current:
        chunks.append("".join(current))

    return chunks

In [3]:
with open("output/wiki/issues.md") as f:
    issues_md = f.read()

chunks = split_markdown(issues_md)

In [4]:
with open("output/wiki/uno.md") as f:
    uno_md = f.read()

tokens = MarkdownIt().parse(uno_md)
set(token.type for token in tokens)

{'bullet_list_close',
 'bullet_list_open',
 'heading_close',
 'heading_open',
 'inline',
 'list_item_close',
 'list_item_open',
 'paragraph_close',
 'paragraph_open'}

In [5]:
from dataclasses import dataclass, field
from collections.abc import Iterator
from markdown_it import MarkdownIt


@dataclass
class Heading:
    level: int
    title: str

    @classmethod
    def from_token(cls, tag: str, content: str) -> "Heading":
        level = int(tag[1:])
        return Heading(level=level, title=content)


@dataclass
class Headings:
    lst: list[Heading] = field(default_factory=list)

    def __str__(self) -> str:
        return "\n\n".join(f"{'#' * h.level} {h.title}" for h in self.lst) + "\n"

    def add(self, heading: Heading):
        # Remove any headings at the same or deeper level
        while self.lst and self.lst[-1].level >= heading.level:
            self.lst.pop()
        self.lst.append(heading)

    def merge(self, other: "Headings") -> "Headings":
        """Merge two Headings lists, keeping the deepest headings."""
        res = Headings(lst=self.lst.copy())
        for heading in other.lst:
            res.add(heading)
        return res


@dataclass
class Segment:
    """A text segment with all the headings in it."""

    headings: Headings = field(default_factory=Headings)
    text: str = ""


def iter_markdown_segments(txt: str) -> Iterator[Segment]:
    lines = txt.splitlines(keepends=True)
    tokens = MarkdownIt().parse(txt)
    segment_headings: Headings = Headings()
    segment_begin = 0
    i = 0

    while i < len(tokens):
        match tokens[i].type:
            case "heading_open":
                h = Heading.from_token(tag=tokens[i].tag, content=tokens[i + 1].content)
                segment_headings.add(h)
                i += 2

            case "paragraph_open":
                _, end = tokens[i].map or []
                # If the inline element is empty, skip it
                if tokens[i + 1].content.strip() == "":
                    i += 3
                    continue

                text = "".join(lines[segment_begin:end])
                yield Segment(headings=segment_headings, text=text)
                i += 2
                segment_begin = end
                segment_headings = Headings()

            case "fence":
                _, end = tokens[i].map or []
                text = "".join(lines[segment_begin:end])
                yield Segment(headings=segment_headings, text=text)
                i += 1
                segment_begin = end
                segment_headings = Headings()

            case _:
                i += 1


# Tests
md = """# This is the title

Body text with **bold** and _italic_ and a [link](http://example.com).

## This is a subheading

### This is a sub-subheading

Another paragraph, with:

- a bullet point
- another bullet point
  - a nested bullet point
- last bullet point

```java
some code
```

A paragraph
"""

[s for s in iter_markdown_segments(md)]

[Segment(headings=Headings(lst=[Heading(level=1, title='This is the title')]), text='# This is the title\n\nBody text with **bold** and _italic_ and a [link](http://example.com).\n'),
 Segment(headings=Headings(lst=[Heading(level=2, title='This is a subheading'), Heading(level=3, title='This is a sub-subheading')]), text='\n## This is a subheading\n\n### This is a sub-subheading\n\nAnother paragraph, with:\n'),
 Segment(headings=Headings(lst=[]), text='\n- a bullet point\n'),
 Segment(headings=Headings(lst=[]), text='- another bullet point\n'),
 Segment(headings=Headings(lst=[]), text='  - a nested bullet point\n'),
 Segment(headings=Headings(lst=[]), text='- last bullet point\n'),
 Segment(headings=Headings(lst=[]), text='\n```java\nsome code\n```\n'),
 Segment(headings=Headings(lst=[]), text='\nA paragraph\n')]

In [6]:
def compact_markdown_segments(
    seq: Iterator[Segment],
    min_length: int = 256,
    max_length: int = 2048,
) -> list[Segment]:
    compacted = []
    curr = Segment()

    for seg in seq:
        if len(curr.text) < min_length or len(curr.text) + len(seg.text) <= max_length:
            # merge segments if the current is too short,
            # or if the combined length is within limits
            curr.text += seg.text
            curr.headings = curr.headings.merge(seg.headings)
        else:
            compacted.append(curr)
            curr = Segment(
                headings=curr.headings.merge(seg.headings),
                text=str(curr.headings) + seg.text,
            )

    if len(curr.text) > min_length:
        compacted.append(curr)

    return compacted


def markdown_segments(
    txt: str, min_length: int = 256, max_length: int = 2048
) -> list[str]:
    return [
        seg.text
        for seg in compact_markdown_segments(
            iter_markdown_segments(txt), min_length, max_length
        )
    ]


markdown_segments(md, min_length=10, max_length=128)

['# This is the title\n\nBody text with **bold** and _italic_ and a [link](http://example.com).\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n\nAnother paragraph, with:\n\n- a bullet point\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n- another bullet point\n  - a nested bullet point\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n- last bullet point\n\n```java\nsome code\n```\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n\nA paragraph\n']

In [7]:
with open("output/wiki/uno.md") as f:
    uno_md = f.read()

uno_seg = markdown_segments(uno_md, min_length=128, max_length=1024)
uno_seg

["# Uno (personaggio)\n\nUno è uno dei personaggi principali della serie a fumetti Disney Italia di PK (PK - Paperinik New Adventures, PK², PK - Pikappa e PK - Nuova era).\n\n## Biografia\n\nSi tratta di un'intelligenza artificiale creata da Everett Ducklair. È in grado di controllare in ogni sua parte la Ducklair Tower, che ne costituisce il corpo centrale (tanto da presentarsi a Paperinik dicendo di essere a tutti gli effetti il palazzo), di svolgere calcoli di estrema difficoltà e di introdursi grazie alle sue superiori capacità nelle reti informatiche convenzionali più protette. Controlla anche la grande parte dei marchingegni costruiti da Ducklair.\n",
 "# Uno (personaggio)\n\n## Biografia\n\nÈ il più valido aiuto di Paperinik, di cui conosce l'identità segreta. L'aspetto con cui si presenta agli umani è molto simile al viso di Everett Ducklair, solitamente presentato in forma di ologramma all'interno di una sfera verde. Il suo acerrimo nemico è Due, la crudele intelligenza artifi

## Prepare dataset

In [8]:
# Deduplicate documents based on their content
hashes = set()
filtered_docs = []

for path in INPUT_DOCS:
    content = path.read_text()
    content_hash = hash(content)
    if content_hash not in hashes:
        hashes.add(content_hash)
        filtered_docs.append(path)

len(INPUT_DOCS), len(filtered_docs)

(262, 216)

In [9]:
original_docs = [path.read_text() for path in filtered_docs]

In [10]:
max([len(doc) for doc in original_docs])

166287

In [11]:
average_length = sum(len(doc) for doc in original_docs) / len(original_docs)
average_length

6262.37962962963

## Rephraser model

Rephrase wiki contents so that it seems to be coming from the comc itself.

In [20]:
import json
import os
from dotenv import load_dotenv
import dspy

load_dotenv()


def load_creds() -> str:
    with open(".env.sa.json") as f:
        sa = json.load(f)
    return json.dumps(sa)


lm = dspy.LM(
    model="vertex_ai/gemini-3-flash-preview",
    vertex_credentials=load_creds(),
    vertex_project=os.getenv("GOOGLE_CLOUD_PROJECT"),
    vertex_location=os.getenv("GOOGLE_CLOUD_LOCATION"),
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_tokens=65535,
)
dspy.configure(lm=lm, track_usage=True)

In [21]:
import dspy


class Rephrase(dspy.Signature):
    """Rephrase the given text about a comic as if it was written WITHIN the fictional universe of the comic.

    Follow this instructions:
    1. Skip information about who wrote the comic.
    2. Focus on the events and characters within the comic.
    3. Maintain the informational content of the original text, when it explains the comic's plot, characters and themes.
    4. Do not translate the text into another language. Keep the original language.
    """

    document: str = dspy.InputField(desc="The document to rephrase.")
    rephrased: str = dspy.OutputField(
        desc="The rephrased document.",
    )


class SyntheticRephraser(dspy.Module):
    """Rephrase the given document."""

    def __init__(self):
        self.rephrase = dspy.Predict(Rephrase)

    def forward(self, doc: str) -> dspy.Prediction:
        return self.rephrase(document=doc)


rephraser = SyntheticRephraser()

In [22]:
# Vibe-check
input_doc = Path("output/wiki/lyla.md").read_text()
res = rephraser(doc=input_doc)

In [23]:
res.get_lm_usage()
rephraser.dump_state()

{'rephrase': {'traces': [],
  'train': [],
  'demos': [],
  'signature': {'instructions': "Rephrase the given text about a comic as if it was written WITHIN the fictional universe of the comic.\n\nFollow this instructions:\n1. Skip information about who wrote the comic.\n2. Focus on the events and characters within the comic.\n3. Maintain the informational content of the original text, when it explains the comic's plot, characters and themes.\n4. Do not translate the text into another language. Keep the original language.",
   'fields': [{'prefix': 'Document:',
     'description': 'The document to rephrase.'},
    {'prefix': 'Rephrased:', 'description': 'The rephrased document.'}]},
  'lm': None}}

In [24]:
print(res.rephrased)

# Dossier Personale: Lyla Lay

Lyla Lay è una figura centrale nella difesa di Paperopoli, nota ufficialmente come una brillante giornalista di 00 Channel, ma la cui vera natura è molto più complessa e legata ai segreti dello spazio-tempo.

### Origini e Missione Operativa

Sebbene appaia come una giovane donna dinamica e intraprendente, Lyla è in realtà un androide di classe 5Y proveniente dal XXIII secolo. In qualità di agente della Tempolizia, il corpo di polizia temporale del futuro, la sua missione primaria consiste nel sorvegliare il corretto scorrimento del flusso temporale nella Paperopoli del XX secolo, intervenendo per neutralizzare i cronopirati. Tra i suoi avversari storici spicca il Razziatore, un temibile predone spazio-temporale.

Nel corso delle sue missioni, Lyla ha stretto una solida e inossidabile alleanza con il guardiano di Paperopoli, Paperinik, del quale conosce la vera identità. La sua fisionomia può apparire variabile: a volte mostra linee eleganti e slanciate, 

In [26]:
# Rephrase all documents
from pathlib import Path
from tqdm.notebook import tqdm

MAX_DOC_LENGTH = 65535 / 2


def rephrase_doc(input_path: Path, output_dir: Path) -> None:
    output_path = output_dir / input_path.relative_to(INPUT_ROOT_DIR)
    stats_path = output_dir / "stats.jsonl"
    model_state_path = output_dir / "model.json"
    if output_path.exists():
        return

    # Skip docs that are too long
    text = input_path.read_text()
    if len(text) > MAX_DOC_LENGTH:
        print(f"Skipping {input_path} with length {len(text)}")
        return

    # Dump model state
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not model_state_path.exists():
        with open(model_state_path, "w") as f:
            json.dump(rephraser.dump_state(), f, indent=2)

    # Generate
    res = rephraser(doc=text)
    with open(output_path, "w") as f:
        f.write(res.rephrased)
    # Update stats
    with open(stats_path, "a") as f:
        f.write(
            json.dumps(
                {
                    "input": str(input_path),
                    "output": str(output_path),
                    "lm_stats": res.get_lm_usage(),
                }
            )
            + "\n"
        )


for path in tqdm(filtered_docs, desc="Rephrasing documents"):
    rephrase_doc(path, OUTPUT_DIR / "rephrased")

Rephrasing documents:   0%|          | 0/216 [00:00<?, ?it/s]

Skipping output/wiki/fandom/crawl/personaggi/evroniani/evroniani.md with length 46115
Skipping output/wiki/issues.md with length 166287
Skipping output/wiki/quotes.md with length 84177


## Q/A generator model

In [18]:
import dspy
from pydantic import BaseModel


class QuestionAnswerPair(BaseModel):
    """A question and its corresponding answer."""

    question: str
    answer: str


class QAGenerator(dspy.Signature):
    """Read the text, ask questions and answer them.

    Follow these instructions:
    1. Ask diverse questions that require different cognitive skills or cover different aspects of the text.
    2. Ask questions in various forms such as:
    - Yes/No questions that require determining whether a statement is true or false.
    - Open-ended questions that begin with words like what, how, when, where, why and who.
    - Multi-choice questions that offers two or more options to choose from. Include the options in the question.
    - Comparison questions that compare two quantities or objects and determine the relationship between them.
    - Reading comprehension questions that test the ability to understand and analyze the text.
    - Problem-solving questions that test the ability to solve mathematical, physical, or logical problems.
    - Long-form questions that require detailed answers or explanations.
    3. Use the document summary and previous question-answer pairs as context.
    - Do not repeat questions that have already been asked, or that are too similar to previously asked questions.
    - New questions should be relevant only to the current document segment.
    - New questions should follow naturally from the previous questions and answers.
    4. Questions should be self-contained and make sense WITHOUT the context given to generate it.
    - Do not use sentences like: "according to the text", or "in the segment".
    - If any context is necessary to understand the question, put it in the question itself.
    5. Focus on asking questions about factual information, important knowledge, or concrete details in the text.
    6. Write questions and answers using clear and concise language.
    7. Use plain text. Do not use Markdown.
    8. Do NOT translate the text into another language. Keep the original language (Italian)."""

    summary: str = dspy.InputField(
        desc="A brief summary of the whole document, for context."
    )
    previous_qa_pairs: list[QuestionAnswerPair] = dspy.InputField(
        desc="A list of previously asked question-answer pairs for the previous segment of the document."
    )
    document_segment: str = dspy.InputField(
        desc="The document segment to ask questions about."
    )
    question_answers: list[QuestionAnswerPair] = dspy.OutputField(
        desc="A list of question-answer pairs.",
        max_length=8,
    )


class SyntheticQAGenerator(dspy.Module):
    """Generate synthetic question-answer pairs from text segments."""

    def __init__(self):
        self.summarizer = dspy.Predict(
            dspy.make_signature(
                signature="text -> summary",
                instructions="Summarize the following text in a concise manner. Keep the original language.",
            )
        )
        self.qa_generator = dspy.ChainOfThought(QAGenerator)

    def forward(self, doc: str, segments: list[str]) -> dspy.Prediction:
        summary = self.summarizer(text=doc).summary

        res = []
        prev = []
        for segment in segments:
            qa_pairs = self.qa_generator(
                summary=summary, previous_qa_pairs=prev, document_segment=segment
            ).question_answers
            res.extend(qa_pairs)
            prev = qa_pairs

        return dspy.Prediction(summary=summary, qa_pairs=res)


qa_generator = SyntheticQAGenerator()

In [19]:
# Vibe check
input_doc = Path("output/synthetic/v2/rephrased/lyla.md").read_text()
segments = markdown_segments(
    input_doc, min_length=MIN_CHUNK_SIZE, max_length=CHUNK_SIZE
)

res = qa_generator(doc=input_doc, segments=segments)

In [20]:
len(segments), len(res.qa_pairs)

(7, 47)

In [21]:
print(res.summary)
for qa in res.qa_pairs:
    print(qa)

Lyla Lay è una figura centrale nella Saga di Pikappa, un droide inizialmente presentata come agente d'élite della Tempolizia sotto copertura come giornalista al 00 Channel, alleata di Paperinik. Nel corso delle diverse fasi della saga (PKNA, PK², Pikappa, PKNE), la sua storia e le sue origini sono evolute: da agente intrappolata nel futuro e poi tornata, a caporedattore del 00 News che ferma minacce temporali, fino a una revisione che la descrive come un droide del XIX secolo, riparato e riattivato. Ha sempre mantenuto il suo ruolo di fedele alleata di Pikappa, operando come giornalista (anche web-giornalista e reporter internazionale) e sviluppando relazioni personali, pur mantenendo la sua natura di droide con manifestazioni fisiche variabili.
question="What was Lyla Lay's initial role during her first appearances in the Pikappa Saga?" answer='Lyla Lay was an elite droid agent of the Tempolizia.'
question='How did Lyla Lay operate under cover to fulfill her mission of safeguarding th

In [22]:
# Generate QA for all documents
import time
from pathlib import Path
from tqdm.notebook import tqdm


def generate_qa(input_path: Path) -> None:
    output_dir = OUTPUT_DIR / "qa"
    output_path = output_dir / input_path.relative_to(
        OUTPUT_DIR / "rephrased"
    ).with_suffix(".json")
    if output_path.exists():
        return

    output_path.parent.mkdir(parents=True, exist_ok=True)
    text = input_path.read_text()
    segments = markdown_segments(text, min_length=MIN_CHUNK_SIZE, max_length=CHUNK_SIZE)

    # Generate
    start = time.time()
    res = qa_generator(doc=text, segments=segments)
    elapsed = time.time() - start

    # Save
    with open(output_path, "w") as f:
        f.write(
            json.dumps(
                {
                    "summary": res.summary,
                    "qa_pairs": [qa.dict() for qa in res.qa_pairs],
                    "input": str(input_path),
                    "lm_stats": res.get_lm_usage(),
                    "elapsed_sec": f"{elapsed:.2f}",
                    "model_state": qa_generator.dump_state(),
                },
                indent=2,
                ensure_ascii=False,
            )
        )


input_docs = [path for path in (OUTPUT_DIR / "rephrased").glob("**/*.md")]
for path in tqdm(input_docs, desc="Generating QA pairs"):
    generate_qa(path)

Generating QA pairs:   0%|          | 0/201 [00:00<?, ?it/s]

## Multi-turn

Generate multi-turn conversations starting from the same rephrased documents.

In [23]:
def load_bio(name: str) -> str:
    with open(f"input/bios/{name}.md", encoding="utf-8") as f:
        return f.read().strip()


pk_bio = load_bio("paperinik")
uno_bio = load_bio("uno")

In [24]:
conversation_default_location = "Uno si trova nel piano segreto della Ducklair Tower. PK è a Paperopoli e sta comunicando con Uno tramite un dispositivo di comunicazione segreto."

In [25]:
import dspy
from pydantic import BaseModel


class SpeakerBio(BaseModel):
    """A brief biography of a speaker."""

    name: str
    bio: str


class MultiTurnGenerator(dspy.Signature):
    """Read the text, generate multi-turn conversations about it between two speakers.

    Follow these instructions:
    1. Use the document summary, previous lines of the conversation, and the speakers' biographies as context.
    2. The conversation should focus on the given document segment.
    3. The conversation should be self-contained and make sense WITHOUT the context given to generate it.
    - Do not use sentences like: "according to the text", or "in the segment".
    - If any context is necessary to understand the conversation, put it in the dialogue itself.
    4. The model conversation should be used as a style guide to match.
    - Generate the new conversation as if it was in the same situation.
    - Do not continue the model conversation or refer to it directly.
    5. The conversation should be engaging and dynamic, with the speakers responding to each other's statements.
    6. Each line of dialogue should be concise and to the point, avoiding long monologues.
    7. The first speaker to talk must be Speaker 1.
    8. Use plain text. Do not use Markdown.
    9. Do NOT translate the text into another language. Keep the original language (Italian)."""

    summary: str = dspy.InputField(
        desc="A brief summary of the whole document, for context."
    )
    document_segment: str = dspy.InputField(
        desc="The document segment the conversation should focus on."
    )
    model_conversation: list[str] = dspy.InputField(
        desc="A sample conversation you must match the style of."
    )
    previous_lines: list[str] = dspy.InputField(
        desc="Some of the previous lines of the conversation, for context."
    )
    speaker1: SpeakerBio = dspy.InputField()
    speaker2: SpeakerBio = dspy.InputField()
    location: str = dspy.InputField(
        desc="A description of the location where the conversation takes place."
    )
    conversation: list[str] = dspy.OutputField(
        desc="A list of dialogue lines, alternating between the two speakers.",
        max_length=8,
    )


class SyntheticMultiTurnGenerator(dspy.Module):
    """Generate synthetic multi-turn conversations from text segments."""

    def __init__(self, speakers: tuple[SpeakerBio, SpeakerBio]):
        self.summarizer = dspy.Predict(
            dspy.make_signature(
                signature="text -> summary",
                instructions="Summarize the following text in a concise manner. Keep the original language.",
            )
        )
        self.speakers = speakers
        self.mt_generator = dspy.ChainOfThought(MultiTurnGenerator)

    def forward(
        self,
        doc: str,
        model_conv: list[str],
        location: str,
        segments: list[str],
    ) -> dspy.Prediction:
        summary = self.summarizer(text=doc).summary

        res = []
        prev = []
        for segment in segments:
            conversation = self.mt_generator(
                summary=summary,
                document_segment=segment,
                model_conversation=model_conv,
                previous_lines=prev,
                speaker1=self.speakers[0],
                speaker2=self.speakers[1],
                location=location,
            ).conversation
            res.extend(conversation)
            prev = conversation

        return dspy.Prediction(summary=summary, conversation=res)


mt_generator = SyntheticMultiTurnGenerator(
    speakers=(
        SpeakerBio(
            name="Pikappa",
            bio=pk_bio,
        ),
        SpeakerBio(
            name="Uno",
            bio=uno_bio,
        ),
    )
)

Prepare the model conversations.

In [26]:
import pandas as pd

model_convos_df = pd.read_csv(INPUT_MODEL_CONVERSATIONS)
model_convos_df.head()

,issue,input_pages,characters,conversations
0,PKNA #0,"[""input/pkna/pkna-0/pkna-0-029.jpg"", ""../in...","[""other"", ""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Hai paura di af..."
1,PKNA #0,"[""input/pkna/pkna-0/pkna-0-046.jpg"", ""../in...","[""other"", ""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Ci sei, uno? Pu..."
2,PKNA #0,"[""input/pkna/pkna-0/pkna-0-057.jpg"", ""../in...","[""other"", ""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Intervento prov..."
3,PKNA #0,"[""input/pkna/pkna-0/pkna-0-069.jpg"", ""../in...","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Sempre su...\nE..."
4,PKNA #0/2,"[""input/pkna/pkna-0-2/PK0-2 005.jpg"", ""../i...","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Buongiorno, uno..."


In [27]:
# Filter conversations that have only pk and uno
def chars_set(val):
    return set(json.loads(val))


mask = model_convos_df["characters"].apply(chars_set).map(lambda s: s == {"pk", "uno"})
filtered_model_convos_df = model_convos_df[mask].reset_index(drop=True)
filtered_model_convos_df.head()

,issue,input_pages,characters,conversations
0,PKNA #0,"[""input/pkna/pkna-0/pkna-0-069.jpg"", ""../in...","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Sempre su...\nE..."
1,PKNA #0/2,"[""input/pkna/pkna-0-2/PK0-2 005.jpg"", ""../i...","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Buongiorno, uno..."
2,PKNA #0/2,"[""input/pkna/pkna-0-2/PK0-2 011.jpg""]","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Stavolta però n..."
3,PKNA #0/2,"[""input/pkna/pkna-0-2/PK0-2 028.jpg""]","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Credi che Lyla ..."
4,PKNA #0/2,"[""input/pkna/pkna-0-2/PK0-2 045.jpg""]","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""E con cio?""}, {..."


In [28]:
model_convos_roles = [
    json.loads(row["conversations"]) for _, row in filtered_model_convos_df.iterrows()
]
model_convos_roles[0]

[{'role': 'human',
  'content': 'Sempre su...\nEhi, aspetta! Non spegnere!\nVoglio vedere la seicentonovantaduesima puntata di "Cuori di silicio"!\nAccomodati. Io ne ho fin sopra le piume, della TV!\nBah! Rischio la vita per difendere Paperopoli, e questo è il ringraziamento!\nSu col morale, vecchio mantello tarlato!\nGrazie per il tentativo di fare dello spirito, uno. Ma non serve.'},
 {'role': 'assistant',
  'content': 'Dai! Mi fai un sorriso, o devo produrre una simulazione?'},
 {'role': 'human',
  'content': "Lascia perdere. Vado sul tetto, ho bisogno d'aria.\nCosa? sul tetto?!\nMeglio di no! Fa freddo, a quest'ora: vuoi prenderti un accidente?\nEeeh?!"},
 {'role': 'assistant',
  'content': 'Mi mancava proprio, una mamma cibernetica!\nDammi retta. Non preferisci fare un pisolino?'},
 {'role': 'human',
  'content': 'Forse hai ragione. Sono molto stanco. Sai, quando mi hai fermato...\n...Ho temuto che ci fosse un altro pericolo in vista!'},
 {'role': 'assistant',
  'content': 'Ma no!

In [29]:
model_convos = [[line["content"] for line in convo] for convo in model_convos_roles]
model_convos[0]

['Sempre su...\nEhi, aspetta! Non spegnere!\nVoglio vedere la seicentonovantaduesima puntata di "Cuori di silicio"!\nAccomodati. Io ne ho fin sopra le piume, della TV!\nBah! Rischio la vita per difendere Paperopoli, e questo è il ringraziamento!\nSu col morale, vecchio mantello tarlato!\nGrazie per il tentativo di fare dello spirito, uno. Ma non serve.',
 'Dai! Mi fai un sorriso, o devo produrre una simulazione?',
 "Lascia perdere. Vado sul tetto, ho bisogno d'aria.\nCosa? sul tetto?!\nMeglio di no! Fa freddo, a quest'ora: vuoi prenderti un accidente?\nEeeh?!",
 'Mi mancava proprio, una mamma cibernetica!\nDammi retta. Non preferisci fare un pisolino?',
 'Forse hai ragione. Sono molto stanco. Sai, quando mi hai fermato...\n...Ho temuto che ci fosse un altro pericolo in vista!',
 'Ma no! Cosa ti viene in men te?\nCaccia Gramon a controllo Evron. Nessuna traccia del terrestre mascherato!\nVai pure a dormire. Fuori è tutto tranquillo.']

In [30]:
# Vibe check
input_doc = Path("output/synthetic/v2/rephrased/lyla.md").read_text()
segments = markdown_segments(
    input_doc, min_length=MIN_CHUNK_SIZE, max_length=CHUNK_SIZE
)

res = mt_generator(
    doc=input_doc,
    model_conv=model_convos[0],
    location=conversation_default_location,
    segments=segments,
)
res

Prediction(
    summary="Lyla Lay è una figura centrale nella Saga di Pikappa, un droide inizialmente presentata come agente d'élite della Tempolizia sotto copertura come giornalista al 00 Channel, alleata di Paperinik. Nel corso delle diverse fasi della saga (PKNA, PK², Pikappa, PKNE), la sua storia e le sue origini sono evolute: da agente intrappolata nel futuro e poi tornata, a caporedattore del 00 News che ferma minacce temporali, fino a una revisione che la descrive come un droide del XIX secolo, riparato e riattivato. Ha sempre mantenuto il suo ruolo di fedele alleata di Pikappa, operando come giornalista (anche web-giornalista e reporter internazionale) e sviluppando relazioni personali, pur mantenendo la sua natura di droide con manifestazioni fisiche variabili.",
    conversation=['Sai, Uno, stavo pensando a Lyla Lay. Quanto ne sai davvero di lei?', "Lyla Lay è una figura centrale. Agente droide d'élite della Tempolizia, con missione di salvaguardare il flusso temporale.", "Gi

Now, match input documents with model conversations deterministically.

In [31]:
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

NUM_ITERS = 10


@dataclass
class MultiTurnInput:
    doc_path: Path
    iter_num: int
    model_convo: list[str]


def mt_input_generator() -> Iterator[MultiTurnInput]:
    rnd = random.Random(42)
    models = []

    input_docs = [path for path in (OUTPUT_DIR / "rephrased").glob("**/*.md")]
    for doc in list(input_docs):
        for i in range(NUM_ITERS):
            if not models:
                models = model_convos.copy()
                rnd.shuffle(models)
            model, models = models[0], models[1:]

            yield MultiTurnInput(
                doc_path=doc,
                iter_num=i,
                model_convo=model,
            )


mt_inputs = list(mt_input_generator())
# Test determinism
assert mt_inputs == list(mt_input_generator())
len(mt_inputs)

2010

Generate all

In [32]:
import json
from tqdm.notebook import tqdm


def generate_multi_turn(mtin: MultiTurnInput) -> None:
    input_path = mtin.doc_path
    output_dir = OUTPUT_DIR / "multi_turn"
    output_path = output_dir / input_path.relative_to(
        OUTPUT_DIR / "rephrased"
    ).with_name(f"{input_path.stem}-{mtin.iter_num}.json")
    if output_path.exists():
        return

    output_path.parent.mkdir(parents=True, exist_ok=True)
    text = input_path.read_text()
    segments = markdown_segments(text, min_length=MIN_CHUNK_SIZE, max_length=CHUNK_SIZE)

    # Generate
    start = time.time()
    res = mt_generator(
        doc=text,
        model_conv=mtin.model_convo,
        location=conversation_default_location,
        segments=segments,
    )
    elapsed = time.time() - start

    # Save
    with open(output_path, "w") as f:
        f.write(
            json.dumps(
                {
                    "summary": res.summary,
                    "conversation": res.conversation,
                    "input": str(input_path),
                    "model_conversation": mtin.model_convo,
                    "location": conversation_default_location,
                    "lm_stats": res.get_lm_usage(),
                    "elapsed_sec": f"{elapsed:.2f}",
                    "model_state": mt_generator.dump_state(),
                },
                indent=2,
                ensure_ascii=False,
            )
        )


for mtin in tqdm(mt_inputs, desc="Generating multi-turn conversations"):
    generate_multi_turn(mtin)

Generating multi-turn conversations:   0%|          | 0/2010 [00:00<?, ?it/s]